# 8×8 Wafer Map 최종 하드웨어 추론

이 노트북은 학습 코드를 제외하고 **새 컴퓨터에서 추론만 실행**하도록 정리했습니다.

최종 경로: `8×8 → V2 최종 분류 → V3 REVIEW 보조 → 방향 후처리 → 출력`

## 1. 패키지 설치

아래 셀은 현재 노트북이 `wafer_final_package` 폴더 안에 있을 때 실행합니다.

In [ ]:
%pip install -r requirements.txt

## 2. 모델 파일 확인

`models/wafer_cnn_v2_best.pt`와 `models/wafer_hierarchical_cnn_v3_best.pt`가 필요합니다.

In [ ]:
from pathlib import Path

required = [
    Path("models/wafer_cnn_v2_best.pt"),
    Path("models/wafer_hierarchical_cnn_v3_best.pt"),
]

for path in required:
    print(path, "->", path.exists())

assert all(path.exists() for path in required), (
    "models 폴더에 .pt 파일 두 개를 넣어주세요."
)

## 3. 모델 로드

In [ ]:
from wafer_model import WaferInferenceSystem
from app import parse_hardware_line, print_result

system = WaferInferenceSystem(
    model_dir="models",
    config_path="config.json",
)

print("모델 로드 완료")
print("device:", system.device)
print("V3 binary threshold:", system.v3_binary_threshold)

## 4. 수동 8×8 테스트

In [ ]:
test_map = [
    [0,0,1,1,1,1,0,0],
    [0,1,1,1,1,1,1,0],
    [1,1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1,1],
    [1,1,1,1,2,2,2,1],
    [1,1,1,1,2,2,2,1],
    [0,1,1,1,2,2,2,0],
    [0,0,1,1,1,1,0,0],
]

result = system.predict(test_map)
print_result(result, show_map=test_map)

## 5. Arduino 포트 확인

포트가 안 보이면 USB 케이블/드라이버/Arduino 연결을 먼저 확인합니다.

In [ ]:
from app import list_serial_ports
list_serial_ports()

## 6. 실시간 시리얼 추론

`SERIAL_PORT`만 자신의 환경에 맞게 변경하세요.

- Windows: `COM3`
- macOS: `/dev/cu.usbmodem...`
- Linux: `/dev/ttyACM0`

In [ ]:
from app import run_serial

SERIAL_PORT = "auto"   # 예: "COM3"
BAUD_RATE = 115200

run_serial(
    system=system,
    port=SERIAL_PORT,
    baud=BAUD_RATE,
    timeout=1.0,
    verbose=False,
)